---
title: "Prediction"
subtitle: "From Excel to Python"
execute:
  echo: true
---

**In this lesson**

- Explaining against predicting: a different question, scored a different way
- A target and its features: can a trip's pass type be guessed from the trip?
- The train/test split, and why a model is never scored on the rows it learned from
- The dumbest model, and the accuracy any real model has to beat
- Logistic regression: a probability per trip, and coefficients as odds
- The confusion matrix: four numbers that accuracy hides
- Moving the threshold: precision against recall, and who decides
- A decision tree: rules you can read, and the depth at which it starts to memorize
- Accuracy and its limits: `model_metrics` for the ROC curve, the scorecard, and calibration
- Where Excel stops for good

**The one idea**

> Prediction is scored on rows the model has never seen, against the guess you could have made without it. A model that is 73% accurate when the majority answer alone gets 61% has earned twelve points; a model that is 69% accurate when the majority gets 69% has earned nothing, however it was built. Always report the baseline next to the score.

::: {.content-visible when-format="html"}
[Download this notebook](chapter_09_lecture.ipynb){download="chapter_09_lecture.ipynb"} &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/lshpaner/excel_to_python_exercises/blob/main/notebooks/chapter_09/chapter_09_lecture.ipynb){.colab-link}
:::

::: {.content-visible unless-format="html"}
Notebook and exercises for this chapter: `github.com/lshpaner/excel_to_python_exercises`, folder `notebooks/chapter_09/`.
:::

## Back to the decision

Chapter 8 explained: it took trips per day and asked what the slope and the weekday coefficients mean. This chapter predicts, which is a different job. Metro has a practical use for it. Most of the system's trips are taken by Monthly Pass holders, and the pay-per-ride riders who behave like pass holders, short weekday trips on electric bikes, are the ones a pass offer would convert. To find them you need a model that looks at a trip and says how likely it is to be a pass holder's. Whether the model's coefficients are interpretable matters less than whether its guesses are right on trips it has not seen.

That last clause is the discipline. Everything in the chapter follows from scoring a model on held-out rows and comparing the score with the guess you could have made for free.

In [ ]:
try:
    import model_metrics
except ImportError:                      # Colab does not have it yet; local setups install it from requirements.txt
    %pip install -q model_metrics
    import model_metrics

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score
from model_metrics import (show_confusion_matrix, plot_threshold_metrics, show_roc_curve,
                           show_calibration_curve, summarize_model_performance)

LOCAL = "../../data/raw/"
URL = ("https://github.com/lshpaner/excel_to_python_exercises/"
       "releases/download/data/")
src = LOCAL if os.path.exists(LOCAL + "metro_trips.csv") else URL

trips = pd.read_csv(src + "metro_trips.csv")
trips["start_time"] = pd.to_datetime(trips["start_time"])
trips = trips[(trips["duration"] > 1) & (trips["duration"] < 1440)].copy()   # chapter 3's keep mask

BLUE, ORANGE, LIGHT = "#1F4E79", "#C55A11", "#9DB4CC"
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams.update({"font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
                     "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "figure.titlesize": 11})
len(trips)

`scikit-learn` [@pedregosa2011] is the library for fitting (it is imported as `sklearn`). Where `statsmodels` reports a model as a table of coefficients with standard errors, scikit-learn reports it as predictions and scores, and every model in it is used the same way: build it, `fit` it on training rows, `predict` on test rows. `model_metrics` [@shpaner2025mm] (by this book's author, like `eda_toolkit`) is the library for scoring: hand it a fitted model and the test rows and it draws the confusion matrix, the threshold curves, the ROC curve and the scorecard that this chapter reads.

## A target and its features

The **target** is the column to predict: 1 if the trip was a Monthly Pass trip, 0 if not. The **features** are the columns the model is allowed to look at, and the rule for choosing them is that each has to be known at the moment you would want the prediction. The pass type itself is obviously out; everything the dock records when a bike leaves and returns is in.

In [ ]:
trips["monthly"] = (trips["passholder_type"] == "Monthly Pass").astype(int)
trips["log_duration"] = np.log(trips["duration"])
trips["hour"] = trips["start_time"].dt.hour
trips["weekend"] = (trips["start_time"].dt.weekday >= 5).astype(int)
trips["electric"] = (trips["bike_type"] == "electric").astype(int)
trips["round_trip"] = (trips["start_station"] == trips["end_station"]).astype(int)

features = ["log_duration", "hour", "weekend", "electric", "round_trip"]
print(f"share of Monthly Pass trips: {trips['monthly'].mean():.1%}")
trips.groupby("monthly")[["duration", "hour", "weekend", "electric", "round_trip"]].mean().round(2)

Before any model is fitted, look at the target and at the features. The target is unbalanced: 61% of trips are Monthly Pass, so a coin flip is the wrong mental baseline. And the feature table already shows which columns will do the work. Monthly Pass trips are shorter (18 minutes against 47 on average), twice as likely to be on an electric bike (57% against 28%), and less likely to be weekend or round trips. The hour of day barely differs, so expect it to contribute little.

Duration goes in as a log, as in chapter 6, so that the difference between 5 and 10 minutes counts as much as the difference between 50 and 100. Each 0/1 column is a category turned into a number by hand, the same trick as chapter 8's weekday dummies.

## The train/test split

A model scored on the rows it learned from will look better than it is, because it has seen the answers. So the rows are split before anything is fitted: 80% to learn from, 20% held back to score on, with the held-back rows chosen at random and the Monthly Pass share kept equal in both (`stratify`). `random_state` fixes the draw so the numbers in this chapter reproduce.

In [ ]:
X = trips[features]
y = trips["monthly"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"train {len(X_train):,} rows, test {len(X_test):,} rows")
print(f"Monthly Pass share in test: {y_test.mean():.1%}")

The test rows are not touched again until a model is finished. Looking at them while building the model, even to decide which features to keep, leaks the answers in through the side door.

## The dumbest model

Before fitting anything, score the model that ignores the features and says "Monthly Pass" for every trip.

In [ ]:
baseline = accuracy_score(y_test, np.ones(len(y_test)))
print(f"always predict Monthly Pass: {baseline:.1%} accurate")

61.4%. That is the number every real model has to beat, and the number to print next to every accuracy in the chapter. A model that reaches 65% has found a little; one at 61% has found nothing, no matter how it was built.

## Logistic regression

Chapter 8's line predicted a number. Logistic regression predicts a probability: it fits a line to the features the same way, then squeezes the line's output through a curve so it lands between 0 and 1. The result, for each trip, is the model's estimate of the chance it was a Monthly Pass trip; the prediction is "yes" when that chance is over one half.

In [ ]:
logit = LogisticRegression(max_iter=1000).fit(X_train, y_train)
prob = logit.predict_proba(X_test)[:, 1]     # one probability per test trip
pred = logit.predict(X_test)                 # 1 where prob >= 0.5
print(f"logistic regression: {accuracy_score(y_test, pred):.1%} accurate on test rows (baseline {baseline:.1%})")

73.1% against a baseline of 61.4%: the features are worth twelve points of accuracy. What the model learned is in its coefficients, and the readable form of a logistic coefficient is the **odds ratio**, $e^{\text{coefficient}}$: how many times the odds of "Monthly Pass" are multiplied when that feature goes up by one.

In [ ]:
pd.Series(np.exp(logit.coef_[0]), index=features, name="odds ratio").round(2)

An electric bike multiplies the odds of a Monthly Pass trip by 4.1, the single strongest signal in the table. A weekend trip multiplies them by 0.75 and a round trip by 0.76, both pointing away from pass holders. The `log_duration` ratio of 0.49 is per unit of log, which is not a unit anyone thinks in; the readable version is that doubling a trip's length multiplies the odds by about 0.6. And `hour` sits at 1.00, confirming what the feature table showed: once you know the duration and the bike, the hour adds nothing.

Odds are easier to feel on example trips. Here are five, and the model's probability for each.

In [ ]:
examples = pd.DataFrame({
    "description": ["10 min, weekday 8am, electric", "10 min, weekday 8am, standard",
                    "45 min, weekend, electric", "45 min, weekend, standard, round trip", "90 min, weekend, standard, round trip"],
    "log_duration": np.log([10, 10, 45, 45, 90]), "hour": [8, 8, 14, 14, 14],
    "weekend": [0, 0, 1, 1, 1], "electric": [1, 0, 1, 0, 0], "round_trip": [0, 0, 0, 1, 1]})
examples["P(Monthly Pass)"] = logit.predict_proba(examples[features])[:, 1].round(2)
examples[["description", "P(Monthly Pass)"]]

The first trip is the commuter profile and the model is 85% sure; the last is a tourist loop and the model is 86% sure the other way. The two in the middle are the point of the exercise: a 45-minute weekend trip on an electric bike is a 60-40 call, and a model that says so is more useful than one that pretends to know.

## The confusion matrix

Accuracy is one number and it hides four. The confusion matrix lays them out: rows are what the trip actually was, columns are what the model said. `show_confusion_matrix` draws it from the model and the test rows.

In [ ]:
#| label: fig-cm
#| out-width: 50%
#| fig-cap: "Confusion matrix for the logistic regression at a threshold of 0.5. Rows are the truth, columns the prediction; the diagonal is right, the off-diagonal is the two kinds of wrong."
show_confusion_matrix(model=logit, X=X_test, y=y_test, model_title="Logistic regression",
                      class_labels=["other", "Monthly Pass"], figsize=(4.5, 4.5), label_fontsize=10, tick_fontsize=9)

Read the diagonal first: 3,448 non-pass trips correctly called "other" (true negatives, TN) and 9,036 Monthly Pass trips correctly called "Monthly Pass" (true positives, TP). The two off-diagonal cells are the two ways to be wrong, and they are not the same mistake. 3,146 other trips were called Monthly Pass (false positives, FP) and 1,457 Monthly Pass trips were called other (false negatives, FN). The model is twice as likely to over-call pass holders as to miss them, which is what you would expect from a model trained on a 61% majority.

Two rates summarize the two errors. **Precision** is the share of the model's "Monthly Pass" calls that were right: $9{,}036 / (9{,}036 + 3{,}146) = 74\%$. **Recall** is the share of actual Monthly Pass trips the model caught: $9{,}036 / (9{,}036 + 1{,}457) = 86\%$.

In [ ]:
print(f"precision {precision_score(y_test, pred):.1%}   recall {recall_score(y_test, pred):.1%}")

The dumbest model has recall of 100% (it calls everything Monthly Pass, so it misses none) and precision of 61%. Any model that trades a little recall for a lot of precision has learned something; this one gave up 14 points of recall for 13 points of precision, which is why its accuracy moved.

## Moving the threshold

The prediction was "Monthly Pass if the probability is at least 0.5", and 0.5 is a default, not a law. Raise it and the model calls fewer trips Monthly Pass, each with more confidence: precision goes up, recall goes down. Lower it and the reverse.

In [ ]:
rows = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    call = (prob >= threshold).astype(int)
    rows.append({"threshold": threshold, "called Monthly Pass": f"{call.mean():.0%}",
                 "accuracy": accuracy_score(y_test, call), "precision": precision_score(y_test, call),
                 "recall": recall_score(y_test, call)})
pd.DataFrame(rows).set_index("threshold").round(3)

The table samples six thresholds; `plot_threshold_metrics` draws every one of them.

In [ ]:
#| label: fig-threshold
#| out-width: 89%
#| fig-cap: "Precision, recall, specificity and F1 for the logistic regression at every threshold from 0 to 1. Recall falls and precision rises as the threshold moves right; the dotted line is the default 0.5. Specificity is recall for the other class (the share of non-pass trips correctly called other), and F1 is the balance of precision and recall."
plot_threshold_metrics(model=logit, X_test=X_test, y_test=y_test, model_title="Logistic regression",
                       figsize=(8, 4.5), label_fontsize=10, tick_fontsize=9)

Which row is right depends on what the prediction is for, and that is a decision for whoever will act on it, not for the model. If Metro is going to mail a pass offer to the riders behind non-pass trips, it wants the model's "other" calls to be clean, so it would set the threshold high and accept that it misses some. If it is sizing the pass-holder market, it wants recall. Accuracy peaks at 0.5 and is the wrong thing to optimize in both cases.

## A decision tree

A logistic regression is one equation. A decision tree [@breiman1984] is a flowchart: at each step it asks one yes-or-no question about one feature, and the leaves are the predictions. It is the model to show a reader who wants to see the rules, and with the raw duration in minutes instead of its log (a tree does not care which; it only compares values to a cutoff) the rules read as English.

In [ ]:
#| label: fig-tree
#| fig-cap: "A depth-3 decision tree for Monthly Pass trips. Each box asks one question; left is yes. The first question is whether the trip is under 32.5 minutes, the second is the bike type, and the leaves report the share of training trips that were Monthly Pass."
X_train_min = X_train.assign(duration=np.exp(X_train["log_duration"])).drop(columns="log_duration")
X_test_min = X_test.assign(duration=np.exp(X_test["log_duration"])).drop(columns="log_duration")
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train_min, y_train)
fig, ax = plt.subplots(figsize=(9, 4.2))
plot_tree(tree, feature_names=list(X_train_min.columns), class_names=["other", "Monthly Pass"],
          filled=True, impurity=False, proportion=True, rounded=True, fontsize=6, ax=ax, precision=2)
plt.tight_layout()
print(f"depth-3 tree: {accuracy_score(y_test, tree.predict(X_test_min)):.1%} accurate on test rows (baseline {baseline:.1%})")

Read the tree from the top. A trip under 32.5 minutes on an electric bike is called Monthly Pass, whatever else is true of it. A trip under 32.5 minutes on a standard bike is Monthly Pass only if it is also under 12.5 minutes. A trip over 32.5 minutes is "other" unless it is on an electric bike and under 71.5 minutes. Those are four rules a dock attendant could apply by eye, and they score 73.0% on the test rows, the same as the logistic regression to a tenth of a point.

### The depth at which a tree memorizes

Depth is the number of questions the tree may ask in a row. More depth means finer rules, and past some point the rules stop describing riders and start describing the training rows.

In [ ]:
#| label: fig-depth
#| out-width: 89%
#| fig-cap: "Accuracy on the training rows and the test rows as the tree is allowed more depth. Training accuracy keeps rising; test accuracy flattens between depth 6 and 8 and falls past depth 10. The gap is memorization."
depths = [1, 2, 3, 4, 6, 8, 10, 15, 20, None]
rows = []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train_min, y_train)
    rows.append({"depth": "none" if d is None else d,
                 "train": accuracy_score(y_train, t.predict(X_train_min)), "test": accuracy_score(y_test, t.predict(X_test_min))})
depth_scores = pd.DataFrame(rows).set_index("depth")
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(range(len(depths)), depth_scores["train"], marker="o", color=LIGHT, label="training rows")
ax.plot(range(len(depths)), depth_scores["test"], marker="o", color=BLUE, label="test rows")
ax.axhline(baseline, color=ORANGE, ls="--", lw=1, label="always Monthly Pass")
ax.set_xticks(range(len(depths))); ax.set_xticklabels(depth_scores.index)
ax.set_xlabel("Maximum depth"); ax.set_ylabel("Accuracy"); ax.set_ylim(0.6, 0.8)
ax.legend(frameon=False, loc="upper left")
ax.set_title("Past depth 8 the tree learns the training rows, not the riders", loc="left")
plt.tight_layout()
depth_scores.round(3).T

With no depth limit the tree is 78% accurate on the rows it learned from and 72.6% on the rows it did not, worse than the depth-3 tree. That gap is the reason for the train/test split. A model scored on its own training rows would have told you to pick the deepest tree, and the deepest tree is the worst one.

## Accuracy and its limits

Both models stop at 73%, and the list of things that number does not say is the chapter's real content.

1. **It is twelve points above the baseline, not 73 points above zero.** On a 61% majority, 73% is a modest model. The same 73% on a 50-50 target would be a strong one. Accuracy means nothing without the base rate beside it.
2. **It is one threshold's number.** The table above has six accuracies for the same model. Report precision and recall at the threshold you will use.
3. **It is the ceiling of these features, not of the question.** The 27% of trips the models get wrong are pass holders who took a long weekend ride and tourists who took a short weekday one, and nothing the dock records separates them. A rider ID, a home station, or a history of past trips would; the columns in this file cannot.
4. **It is one random split.** Draw a different 20% and the number moves by a few tenths of a point. Chapter 7's standard error applies here too, and serious work repeats the split several times and reports the spread.

One more score is worth knowing because it appears in every report of this kind. The **AUC** (area under the ROC curve) [@hanley1982] scores the probabilities rather than the yes-or-no calls: it is the chance that a randomly chosen Monthly Pass trip gets a higher probability than a randomly chosen other trip. A coin gives 0.5, perfection gives 1.0. The ROC curve itself traces every threshold at once, true positive rate (recall) against false positive rate, and a model is as good as the area it encloses above the diagonal.

In [ ]:
#| label: fig-roc
#| out-width: 56%
#| fig-cap: "ROC curves for both models on the test rows. Each curve is every threshold at once; the dashed diagonal is a coin flip. The two models enclose the same area to the second decimal."
tree_log = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)   # the same tree, cutoffs in log units, so both models score on X_test
show_roc_curve(model=[logit, tree_log], X=X_test, y=y_test, model_title=["Logistic regression", "Depth-3 tree"],
               overlay=True, figsize=(5, 5), label_fontsize=10, tick_fontsize=9)

0.74 and 0.75: the models rank trips usefully, and not much better than that. AUC does not depend on the threshold or the base rate, which is why it travels well between problems, and why it is the number to compare two models on before choosing a threshold for either.

### The scorecard

`summarize_model_performance` puts every score in this chapter, and a few more, in one table per model.

In [ ]:
summarize_model_performance(model=[logit, tree_log], X=X_test, y=y_test,
                            model_title=["Logistic regression", "Depth-3 tree"], return_df=True)

Three rows are new. **Specificity** is recall for the other class: the share of non-pass trips correctly called other, 52%, which is the weak side of both models. **F1** is a single number balancing precision and recall. The **Brier score** [@brier1950] is the average squared gap between the predicted probability and the truth (0 or 1), so lower is better; it is the score that checks the probabilities themselves rather than the calls.

### Are the probabilities honest?

The chapter said the probability is the useful product, which raises a question accuracy cannot answer: when the model says 0.7, are 70% of those trips actually Monthly Pass? A calibration curve groups test trips by predicted probability and plots the actual share in each group.

In [ ]:
#| label: fig-calibration
#| out-width: 56%
#| fig-cap: "Calibration of the logistic regression: predicted probability (grouped in ten bins) against the actual Monthly Pass share in each bin. Points on the diagonal are honest probabilities; the model runs close to it from 0.3 to 0.8 and over-promises at the top."
show_calibration_curve(model=logit, X=X_test, y=y_test, model_title="Logistic regression", figsize=(5, 5),
                       label_fontsize=10, tick_fontsize=9)

Between 0.3 and 0.8 the model's probabilities mean what they say, which is what a list of "trips rated above 0.7" needs. Above 0.8 the curve flattens: trips the model rates at 0.9 are Monthly Pass about 80% of the time, so the top of the list is a little less certain than it claims. For Metro's purpose that is good enough, and knowing it is the point.

## Where Excel stops for good

| Task | Excel | Python |
|:--|:--|:--|
| Hold out test rows | sort, cut, paste to a second sheet | `train_test_split` |
| Baseline accuracy | `COUNTIF` | `accuracy_score` against a constant |
| Logistic regression | Solver, with the log-likelihood written by hand | `LogisticRegression().fit()` |
| Decision tree | none | `DecisionTreeClassifier().fit()` |
| Confusion matrix | `COUNTIFS` with four conditions | `confusion_matrix` |
| Precision, recall, AUC, calibration | formulas on the matrix; AUC by hand is an afternoon | `summarize_model_performance`, one call |
| Scoring a new trip | type the equation into a cell | `.predict_proba` |

Chapters 7 and 8 had an Excel column with real entries in it. This one mostly does not, and that is the true shape of the handoff. A logistic regression can be forced out of Solver by someone who knows the likelihood; nobody builds a decision tree in a spreadsheet, and nobody should. The reason to have learned chapters 4 and 5 is that the eight lines of feature engineering at the top of this chapter are the whole bridge between the trips file and a model, and they are pandas.

## Back to the decision

Can a trip's pass type be guessed from the trip? Partly. Duration and bike type carry most of what there is to know, the hour carries nothing, and both models land at 73% against a 61% baseline. The useful product is the probability, not the call: a list of non-pass trips the model rates above 0.7 is a list of riders behaving like pass holders, and that list is what a pass offer should go to. The threshold that defines "above 0.7" is Metro's choice, made from the precision-recall table, and the memo should show that table rather than hide it behind one accuracy.

## Looking ahead

- **Worked exercise:** `chapter_09_guided.ipynb`. The same pipeline with the bike as the target: predict whether a trip was on an electric bike from the trip and the pass type, read the odds ratios, and compare the tree with the logistic regression.
- **On your own:** `chapter_09_solo.ipynb`. Predict whether a trip happened on a weekend. The model will report 69% accuracy; find out why that is nothing, using the confusion matrix, and write the sentence that says so.
- **Further reading:** @james2023 covers every model in this chapter, and several more, at the next level of depth, with Python code.
- **Before the next chapter:** chapter 10 writes the memo. Go back through the "Back to the decision" sections of chapters 1 to 9 and list the findings in one place; the memo is the ones that survive.

## Appendix: reproducing this chapter {.unnumbered}

`chapter_09_lecture.ipynb` is generated from this page; exercises come from `tools/build_chapter_09_exercises.py`. Models are `scikit-learn`'s `LogisticRegression` and `DecisionTreeClassifier`, scored with `model_metrics` (PyPI `model_metrics`, in `requirements.txt`), fitted on an 80/20 stratified split with `random_state=42` (the split) and `random_state=0` (the trees), on clean rows (the chapter 3 `keep` mask) from `data/raw/`.